#Get data from youtube

In [1]:
!pip -q install google-api-python-client youtube-transcript-api yt-dlp isodate pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.0/485.0 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 91.7 MB/s eta 0:00:00


In [4]:

from getpass import getpass
API_KEY = getpass("Paste your YouTube API key (input hidden): ")
PLAYLIST_ID = "PLUGvmK5KUAgtjWMAWJ0kwVRHrDLssJeke&index=3"

Paste your YouTube API key (input hidden): ··········


#Download data

In [3]:
!apt -qq install ffmpeg
!pip -q install yt-dlp faster-whisper nltk pandas opencv-python-headless

ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.7 MB/s eta 0:00:00


In [8]:
!pip -q install yt-dlp
import os, glob
from pathlib import Path
import yt_dlp

# --- CONFIG ---
VIDEO_ID   = "hhLOQ0S-uJo"
VIDEO_URL  = f"https://www.youtube.com/watch?v={VIDEO_ID}"
BASE_DIR   = "/content/emo_sign_single"
VIDEOS_DIR = f"{BASE_DIR}/videos"
os.makedirs(VIDEOS_DIR, exist_ok=True)

# Force download + show errors if any
ydl_opts = {
    "outtmpl": f"{VIDEOS_DIR}/%(id)s.%(ext)s",  # -> /content/emo_sign_single/videos/hhLOQ0S-uJo.mp4 (or .webm)
    "format": "mp4[height<=480]/mp4/best[ext=mp4]/best",  # allow fallback to best if no mp4
    "noplaylist": True,
    "ignoreerrors": False,
    "retries": 10,
    "fragment_retries": 10,
    "concurrent_fragment_downloads": 5,
    "geo_bypass": True,
    "quiet": False,   # show diagnostics in the cell output
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(VIDEO_URL, download=True)

print("yt-dlp returned id/ext:", info.get("id"), info.get("ext"))
candidates = glob.glob(f"{VIDEOS_DIR}/{info.get('id') or VIDEO_ID}.*")
print("Downloaded files:", candidates)
assert candidates, "Download failed. Scroll up for yt-dlp errors (network, age/region block, etc.)."
src = candidates[0]
print("Using src:", src)


[youtube] Extracting URL: https://www.youtube.com/watch?v=hhLOQ0S-uJo
[youtube] hhLOQ0S-uJo: Downloading webpage
[youtube] hhLOQ0S-uJo: Downloading tv simply player API JSON
[youtube] hhLOQ0S-uJo: Downloading tv client config
[youtube] hhLOQ0S-uJo: Downloading tv player API JSON
[youtube] hhLOQ0S-uJo: Downloading player 0004de42-main
[info] hhLOQ0S-uJo: Downloading 1 format(s): 18
[download] Sleeping 2.00 seconds as required by the site...
[download] Destination: /content/emo_sign_single/videos/hhLOQ0S-uJo.mp4
[download] 100% of    3.79MiB in 00:00:00 at 4.61MiB/s   
yt-dlp returned id/ext: hhLOQ0S-uJo mp4
Downloaded files: ['/content/emo_sign_single/videos/hhLOQ0S-uJo.mp4']
Using src: /content/emo_sign_single/videos/hhLOQ0S-uJo.mp4


In [19]:
# EMOTION FROM VIDEO (no segmentation)
!pip -q install fer==22.5.0 mtcnn==0.1.1 opencv-python-headless pandas tqdm

import os, cv2, math, json, glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from fer import FER

# Paths
VIDEO_ID   = "hhLOQ0S-uJo"
BASE_DIR   = "/content/emo_sign_single"
VIDEO_PATH = glob.glob(f"{BASE_DIR}/videos/{VIDEO_ID}.*")[0]

# Settings
SAMPLE_FPS = 2.0  # analyze ~2 frames per second (tweak for speed/quality)

# Init FER (uses Keras under the hood; mtcnn=True is more accurate on faces)
detector = FER(mtcnn=True)

# Read video & sample frames
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "Cannot open video."

fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
duration = frame_count / max(fps, 1e-6)

step = max(1, int(round(fps / SAMPLE_FPS)))  # sample every 'step' frames

records = []
idx = -1
pbar = tqdm(total=frame_count//step + 1, desc="Emotion frames")
while True:
    ok, frame = cap.read()
    if not ok: break
    idx += 1
    if idx % step != 0:
        continue
    t = idx / fps

    # FER expects RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # detect_emotions returns list of {box, emotions:{'angry':..,'disgust':.., ...}}
    dets = detector.detect_emotions(rgb)

    if dets:
        # take the face with max "dominant" prob (sum probs, pick max by any emotion)
        # Simple rule: pick face with highest "happy + surprise + neutral" (proxy for clear face)
        best = max(dets, key=lambda d: max(d["emotions"].values()))
        probs = best["emotions"]
        top_emotion = max(probs, key=probs.get)
        rec = {"time_sec": round(t,3), "top_emotion": top_emotion, **{f"p_{k}": float(v) for k,v in probs.items()}}
    else:
        rec = {"time_sec": round(t,3), "top_emotion": "no_face"}

    records.append(rec)
    pbar.update(1)

cap.release()
pbar.close()

# Save per-frame emotions
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)
frame_csv = f"{BASE_DIR}/outputs/{VIDEO_ID}_frame_emotions.csv"
df = pd.DataFrame(records)
df.to_csv(frame_csv, index=False)
print("Per-frame emotions CSV:", frame_csv)

# Aggregate to video-level summary
emo_cols = [c for c in df.columns if c.startswith("p_")]
summary = {}
if len(df):
    # dominant (mode of non-'no_face')
    mode_series = df.loc[df["top_emotion"]!="no_face","top_emotion"]
    dominant = mode_series.mode().iloc[0] if len(mode_series) else "no_face"
    # mean probabilities over frames where a face was detected
    mean_probs = df.loc[df["top_emotion"]!="no_face", emo_cols].mean().to_dict() if len(mode_series) else {}
    summary = {
        "video_id": VIDEO_ID,
        "duration_sec": duration,
        "frames_sampled": int(len(df)),
        "dominant_emotion": dominant,
        "mean_emotions": mean_probs
    }

summary_json = f"{BASE_DIR}/outputs/{VIDEO_ID}_video_emotion_summary.json"
with open(summary_json, "w") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Video summary JSON:", summary_json)
summary


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 79.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

Emotion frames:   1%|          | 1/199 [00:00<00:51,  3.83it/s]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs

Per-frame emotions CSV: /content/emo_sign_single/outputs/hhLOQ0S-uJo_frame_emotions.csv
Video summary JSON: /content/emo_sign_single/outputs/hhLOQ0S-uJo_video_emotion_summary.json


{'video_id': 'hhLOQ0S-uJo',
 'duration_sec': 99.26583333333333,
 'frames_sampled': 199,
 'dominant_emotion': 'neutral',
 'mean_emotions': {'p_angry': 0.049435897435897436,
  'p_disgust': 0.0011282051282051283,
  'p_fear': 0.0461025641025641,
  'p_happy': 0.1197948717948718,
  'p_sad': 0.08994871794871795,
  'p_surprise': 0.0118974358974359,
  'p_neutral': 0.6799999999999999}}

In [25]:
# OCR SUBTITLES → TRANSCRIPT (no audio needed)
!pip -q install easyocr==1.7.1 rapidfuzz==3.9.6 opencv-python-headless pandas nltk

import os, glob, cv2, math, json, numpy as np, pandas as pd
from pathlib import Path
import easyocr
from rapidfuzz import fuzz

VIDEO_ID   = "hhLOQ0S-uJo"
BASE_DIR   = "/content/emo_sign_single"
VIDEO_PATH = glob.glob(f"{BASE_DIR}/videos/{VIDEO_ID}.*")[0]
OUT_DIR    = f"{BASE_DIR}/outputs"
os.makedirs(OUT_DIR, exist_ok=True)
TRANS_JSON = f"{OUT_DIR}/{VIDEO_ID}_transcripts_ocr.json"

# --- Settings (tweak if needed) ---
SAMPLE_FPS      = 2.0     # analyze ~2 frames/sec
ROI_BOTTOM_FRAC = 0.35    # scan bottom 35% of the frame (typical subtitle area)
CONF_MIN         = 0.55   # min OCR confidence per word
MERGE_SIM        = 85     # min similarity (0-100) to treat consecutive texts as same subtitle
MAX_GAP_SEC      = 0.75   # allow brief gaps (subtitle flicker) when merging
JOIN_DETECTIONS  = True   # if multiple boxes per frame, join texts with spaces

# Init OCR (use GPU if available)
use_gpu = Path("/proc/driver/nvidia/version").exists()
reader = easyocr.Reader(['en'], gpu=use_gpu, verbose=False)

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "Cannot open video."
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
step = max(1, int(round(fps / SAMPLE_FPS)))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
duration = frame_count / max(fps, 1e-6)

samples = []  # (t, text)

idx = -1
while True:
    ok, frame = cap.read()
    if not ok:
        break
    idx += 1
    if idx % step != 0:
        continue
    t = idx / fps

    H, W = frame.shape[:2]
    y0 = int(H * (1.0 - ROI_BOTTOM_FRAC))
    roi = frame[y0:H, 0:W]

    # optional light pre-processing helps a bit
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 5, 40, 40)
    roi_proc = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

    # easyocr returns [ [box, text, conf], ... ]
    detections = reader.readtext(roi_proc, detail=1, paragraph=False, min_size=10,
                                 text_threshold=0.6, low_text=0.3, link_threshold=0.5)
    texts = [d[1].strip() for d in detections if d[2] >= CONF_MIN and d[1].strip()]
    if not texts:
        continue
    text = " ".join(texts) if JOIN_DETECTIONS else max(texts, key=len)

    # basic cleaning
    text = " ".join(text.split())
    if len(text) < 2:
        continue

    samples.append((t, text))

cap.release()

# Group consecutive similar texts into segments
segments = []
if samples:
    current_text = samples[0][1]
    start_t = last_t = samples[0][0]

    for t, txt in samples[1:]:
        sim = fuzz.token_set_ratio(current_text, txt)
        if sim >= MERGE_SIM and (t - last_t) <= MAX_GAP_SEC:
            # same subtitle continues
            last_t = t
            # prefer the longer/cleaner text
            if len(txt) > len(current_text):
                current_text = txt
        else:
            # close previous segment
            end_t = last_t + (1.0 / SAMPLE_FPS)
            segments.append({"text": current_text, "start": float(start_t), "duration": float(max(0.1, end_t - start_t))})
            # start new
            current_text = txt
            start_t = last_t = t

    # close final
    end_t = last_t + (1.0 / SAMPLE_FPS)
    segments.append({"text": current_text, "start": float(start_t), "duration": float(max(0.1, end_t - start_t))})

# Save transcript JSON (even if empty)
with open(TRANS_JSON, "w", encoding="utf-8") as f:
    json.dump({VIDEO_ID: segments}, f, ensure_ascii=False, indent=2)

print(f"OCR transcript segments: {len(segments)}")
print("Saved:", TRANS_JSON)
pd.DataFrame(segments).head(10)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 kB 22.3 MB/s eta 0:00:00


  warnings.warn(warn_msg)



OCR transcript segments: 14
Saved: /content/emo_sign_single/outputs/hhLOQ0S-uJo_transcripts_ocr.json


,text,start,duration
0,Chao cac ban den voi Ban tin Xa hoi. mung,1.0010,2.5020
1,That su rat xuc dong!,4.0040,1.5010
2,"Gan day, thong qua Hoi chu thap do, chinh phu ...",7.0070,4.0035
3,"ung ho nhan dan Cuba voi chu de ""65 nam nghia ...",11.0110,12.0115
4,VND (Chuong trinh dien ra tu ngay 13/8 den het...,23.0230,8.0075
5,VND dat muc tieu van dong toi thieu 65 ty dong:,31.0310,7.0065
6,va sat canh cua nhan dan Cuba trong nhung nam ...,38.5385,0.5000
7,va sat canh cua nhan dan Cuba trong nhung nam ...,39.5395,3.0025
8,"Nam 1966, Ianh tu Fidel Castro tung noi: ""Vi V...",42.5425,8.0075
9,"Trong chien tranh; Cuba da ho tra thuoc men, l...",50.5505,9.5090


In [26]:
# VADER on OCR transcript
import json, pandas as pd, nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

VIDEO_ID   = "hhLOQ0S-uJo"
BASE_DIR   = "/content/emo_sign_single"
TRANS_JSON = f"{BASE_DIR}/outputs/{VIDEO_ID}_transcripts_ocr.json"
SENTS_CSV  = f"{BASE_DIR}/outputs/{VIDEO_ID}_sentiment_vader_ocr.csv"

with open(TRANS_JSON, "r", encoding="utf-8") as f:
    tx = json.load(f)
segments = tx.get(VIDEO_ID, []) or []

sia = SentimentIntensityAnalyzer()

def vader_label(c):
    return "unavailable" if c is None else ("positive" if c>=0.05 else ("negative" if c<=-0.05 else "neutral"))
def intensity(c):
    if c is None: return "unavailable"
    m=abs(c); return "low" if m<0.25 else ("medium" if m<0.55 else "high")

rows=[]
# Per-segment
for s in segments:
    sc = sia.polarity_scores(s["text"])
    rows.append({
        "level": "segment",
        "start_sec": s["start"],
        "duration_sec": s["duration"],
        "text": s["text"],
        "vader_compound": sc["compound"],
        "vader_pos": sc["pos"],
        "vader_neu": sc["neu"],
        "vader_neg": sc["neg"],
        "sentiment_label": vader_label(sc["compound"]),
        "sentiment_intensity": intensity(sc["compound"]),
    })

# Overall
if segments:
    full_text = " ".join(s["text"] for s in segments).strip()
    sc = sia.polarity_scores(full_text)
    rows.insert(0, {
        "level": "overall",
        "start_sec": 0.0,
        "duration_sec": sum(s["duration"] for s in segments),
        "text": full_text[:5000],
        "vader_compound": sc["compound"],
        "vader_pos": sc["pos"],
        "vader_neu": sc["neu"],
        "vader_neg": sc["neg"],
        "sentiment_label": vader_label(sc["compound"]),
        "sentiment_intensity": intensity(sc["compound"]),
    })
else:
    rows.insert(0, {
        "level": "overall",
        "start_sec": 0.0,
        "duration_sec": 0.0,
        "text": "",
        "vader_compound": None, "vader_pos": None, "vader_neu": None, "vader_neg": None,
        "sentiment_label": "unavailable", "sentiment_intensity": "unavailable",
    })

sent_df = pd.DataFrame(rows)
sent_df.to_csv(SENTS_CSV, index=False)
print("Saved:", SENTS_CSV)
sent_df.head(5)


Saved: /content/emo_sign_single/outputs/hhLOQ0S-uJo_sentiment_vader_ocr.csv


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


,level,start_sec,duration_sec,text,vader_compound,vader_pos,vader_neu,vader_neg,sentiment_label,sentiment_intensity
0,overall,0.000,94.5875,Chao cac ban den voi Ban tin Xa hoi. mung That...,-0.8439,0.022,0.922,0.056,negative,high
1,segment,1.001,2.5020,Chao cac ban den voi Ban tin Xa hoi. mung,-0.8020,0.000,0.526,0.474,negative,high
2,segment,4.004,1.5010,That su rat xuc dong!,0.0000,0.000,1.000,0.000,neutral,low
3,segment,7.007,4.0035,"Gan day, thong qua Hoi chu thap do, chinh phu ...",0.0000,0.000,1.000,0.000,neutral,low
4,segment,11.011,12.0115,"ung ho nhan dan Cuba voi chu de ""65 nam nghia ...",0.0000,0.000,1.000,0.000,neutral,low


In [30]:
sent_df["text"]

,text
0,Chao cac ban den voi Ban tin Xa hoi. mung That...
1,Chao cac ban den voi Ban tin Xa hoi. mung
2,That su rat xuc dong!
3,"Gan day, thong qua Hoi chu thap do, chinh phu ..."
4,"ung ho nhan dan Cuba voi chu de ""65 nam nghia ..."
5,VND (Chuong trinh dien ra tu ngay 13/8 den het...
6,VND dat muc tieu van dong toi thieu 65 ty dong:
7,va sat canh cua nhan dan Cuba trong nhung nam ...
8,va sat canh cua nhan dan Cuba trong nhung nam ...
9,"Nam 1966, Ianh tu Fidel Castro tung noi: ""Vi V..."
